**ML-04 — Search Intelligence Data Contract**

**1. Unit of analysis + time window**

    One row = one what, over which dates? State it, then verify it below.

In [ ]:
"one row = one content item, on one report date from fact_content_daily_performance. Time window is developing on month=2026-03."

**2. Fields: feature / label / context / excluded**

    Sort every field you plan to touch into these four buckets. Excluded needs a why.

In [ ]:
"Tables I'll use are fact_content_daily_performance_sample, fact_content_daily_performance, joined with dim_content and dim_clients. I'd predict or rank whether a content item's performance is declining, but computed myself from raw daily rows. I deliberately exclude any GA4 engagement column for rows where ga4_data_available = FALSE"

**3. Verify it with queries (grain, counts, missing values, windows)**

    Every claim above gets a query cell here. A contract claim without a query next to it is a guess.

In [11]:
"Q1:"
DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows violating the grain:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain: 0


,report_date,client_hash_id,content_hash_id,c


In [15]:
"Q2:"
span_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{DEV_MONTH_PATH}')
""").df()
span_check

,n_rows,min_date,max_date,n_content_items
0,9841378,2026-03-01,2026-03-31,331437


In [13]:
"Q3:"
avail_check = con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
    FROM read_parquet('{DEV_MONTH_PATH}')
""").df()
avail_check

,total_rows,rows_with_ga4
0,9841378,413966.0


In [23]:
"five features, max: build a small feature frame for your lane from that same month, and give every feature one line: 'knowable at the decision moment because'"

features = con.sql(f"""
    SELECT
      f.content_hash_id,
      f.gsc_avg_position,
      d.content_type,
      d.word_count,
      DATE_DIFF('day', d.content_created_date, f.report_date) AS content_age_days,
      f.gsc_impressions
    FROM read_parquet('{DEV_MONTH_PATH}') f
    JOIN read_parquet('{DIM_CONTENT}') d
      ON f.content_hash_id = d.content_hash_id
""").df()
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,gsc_avg_position,content_type,word_count,content_age_days,gsc_impressions
0,content_b7e512995f79d5a6,3.350000,keyword article,<NA>,366,20
1,content_05597932fe4da067,0.000000,keyword article,<NA>,366,1
2,content_7a105f548d9c6916,4.928000,keyword article,2123,366,125
3,content_905aa32a0230694e,4.000000,keyword article,<NA>,366,7
4,content_a3ea9792f793ec72,2.272727,keyword article,<NA>,366,11


In [25]:
"THE TRAP:"

labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

df = features.drop_duplicates('content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
).dropna()

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = df[['gsc_avg_position','word_count','content_age_days','gsc_impressions']]
y = df['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
print("Honest AUC (before trap):", honest_auc)

# --- THE TRAP ---
df = df.merge(labels[['content_hash_id','clicks_second_half']], on='content_hash_id')
df_leak = df.dropna(subset=['gsc_avg_position','word_count','content_age_days','gsc_impressions','clicks_second_half'])

X_leak = df_leak[['gsc_avg_position','word_count','content_age_days','gsc_impressions','clicks_second_half']]
y_leak = df_leak['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y_leak, test_size=0.3, random_state=0)
model_leak = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leaked_auc = roc_auc_score(y_te, model_leak.predict_proba(X_te)[:,1])
print("Leaked AUC (with trap column):", leaked_auc)

print(f"\nFinal reported metric: {honest_auc:.3f} (leaked version {leaked_auc:.3f} discarded)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (before trap): 0.6699721798100586
Leaked AUC (with trap column): 0.6989988420011455

Final reported metric: 0.670 (leaked version 0.699 discarded)


**4. Data limits**

    What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.

In [ ]:
"This slice only covers month=2026-03 for one 30-day window. The client history depth varies wildly and a third of clients have little or no usable GA4 history. So any pattern found here should be rechecked against the full fact_content_daily_performance table before being trusted"

**Self-check**

    Before you submit, confirm each line honestly:


✅ Every section above is filled — markdown thinking AND the code that backs it

✅ The notebook runs top to bottom with no errors (Runtime → Run all)

✅ No client names, URLs, or private queries anywhere

✅ My claims use careful words: observed, measured, directional, decision-support

✅ Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.